In [2]:
// 0908

pub struct CircularBuffer<T> {
    buffer: Box<[T]>,
    capacity: usize,
    head: usize,
    tail: usize,
    count: usize,
}

impl<T> CircularBuffer<T>
where
    T: Copy + Default,
{
    pub fn new(capacity: usize) -> Self {
        let buffer = vec![T::default(); capacity].into_boxed_slice();
        Self {
            buffer,
            capacity,
            head: 0,
            tail: 0,
            count: 0,
        }
    }

    pub fn push(&mut self, item: T) -> Option<T> {
        if self.capacity == 0 {
            return Some(item); // drop immediately
        }

        let result = if self.is_full() {
            self.pop()
        } else {
            None
        };

        self.buffer[self.tail] = item;
        self.tail += 1;
        if self.tail == self.capacity {
            self.tail = 0;
        }

        self.count += 1;
        result
    }

    pub fn pop(&mut self) -> Option<T> {
        if self.is_empty() {
            return None;
        }

        let item = self.buffer[self.head];
        self.head += 1;
        if self.head == self.capacity {
            self.head = 0;
        }

        self.count -= 1;
        Some(item)
    }

    pub fn reset(&mut self) {
        self.head = 0;
        self.tail = 0;
        self.count = 0;
    }

    pub fn is_empty(&self) -> bool {
        self.count == 0
    }

    pub fn is_full(&self) -> bool {
        self.capacity == 0 || self.count == self.capacity
    }

    pub fn len(&self) -> usize {
        self.count
    }

    pub fn capacity(&self) -> usize {
        self.capacity
    }

    /// Iterates over values by reference.
    pub fn iter(&self) -> impl Iterator<Item = &T> {
        (0..self.count).map(move |i| {
            let mut idx = self.head + i;
            if idx >= self.capacity {
                idx -= self.capacity;
            }
            &self.buffer[idx]
        })
    }
}



fn main() {
    let mut cb = CircularBuffer::new(5);

    cb.push(10.0);
    cb.push(20.0);
    cb.push(30.0);
    cb.push(40.0);
    cb.push(50.0);

    println!("Initial buffer: {:?}", cb.iter().copied().collect::<Vec<_>>());

    let dropped = cb.push(60.0);
    println!("Pushed 60.0, dropped: {:?}", dropped);
    println!("Current buffer: {:?}", cb.iter().copied().collect::<Vec<_>>());

    println!("Is full? {}", cb.is_full());
    println!("Is empty? {}", cb.is_empty());
    println!("Length: {}", cb.len());
    println!("Capacity: {}", cb.capacity());

    let sum: f64 = cb.iter().copied().sum();
    println!("Sum: {}", sum);

    let min = cb
        .iter()
        .copied()
        .filter(|x| !x.is_nan())
        .min_by(|a, b| a.partial_cmp(b).unwrap());
    println!("Min (ignoring NaN): {:?}", min);

    cb.reset();
    println!("\nAfter reset:");
    println!("Is empty? {}", cb.is_empty());
    println!("Contents: {:?}", cb.iter().copied().collect::<Vec<_>>());

    cb.push(f64::NAN);
    cb.push(f64::NAN);
    println!("\nAfter pushing NaNs:");
    let min_nan = cb
        .iter()
        .copied()
        .filter(|x| !x.is_nan())
        .min_by(|a, b| a.partial_cmp(b).unwrap());
    println!("Min (ignoring NaN): {:?}", min_nan);
}

main()

Initial buffer: [10.0, 20.0, 30.0, 40.0, 50.0]
Pushed 60.0, dropped: Some(10.0)
Current buffer: [20.0, 30.0, 40.0, 50.0, 60.0]
Is full? true
Is empty? false
Length: 5
Capacity: 5
Sum: 200
Min (ignoring NaN): Some(20.0)

After reset:
Is empty? true
Contents: []

After pushing NaNs:
Min (ignoring NaN): None


()